In [3]:
# ============================================================
# CELL 1: Imports + Modell-Konfiguration
# ============================================================
import os
import torch
import numpy as np
import pandas as pd
from PIL import Image, UnidentifiedImageError
from pathlib import Path
from transformers import CLIPProcessor, CLIPModel
import open_clip
import mobileclip
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_score, cross_val_predict
from sklearn.metrics import classification_report, confusion_matrix
import joblib

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Gerät: {device}")

MOBILECLIP_WEIGHTS = r"C:\Users\Lukas\masterthesis\thesis_2026\data\weights\mobileclip_weights\mobileclip_s1.pt"

MODEL_CONFIGS = [
    {"name": "CLIP-ViT-B32",   "type": "hf",         "model_id": "openai/clip-vit-base-patch32"},
    {"name": "CLIP-ViT-L14",   "type": "hf",         "model_id": "openai/clip-vit-large-patch14"},
    {"name": "EVA01-CLIP-g14", "type": "open_clip",  "model_id": "EVA01-g-14", "pretrained": "laion400m_s11b_b41k"},
    {"name": "MobileCLIP-S1",  "type": "mobileclip", "model_id": "mobileclip_s1", "pretrained": MOBILECLIP_WEIGHTS},
]

def load_model(config):
    if config["type"] == "hf":
        processor = CLIPProcessor.from_pretrained(config["model_id"])
        model = CLIPModel.from_pretrained(config["model_id"]).to(device)
        model.eval()
        return {"type": "hf", "model": model, "processor": processor}
    elif config["type"] == "open_clip":
        model, _, preprocess = open_clip.create_model_and_transforms(
            config["model_id"], pretrained=config["pretrained"]
        )
        model = model.to(device).eval()
        return {"type": "open_clip", "model": model, "preprocess": preprocess}
    elif config["type"] == "mobileclip":
        model, _, preprocess = mobileclip.create_model_and_transforms(
            config["model_id"], pretrained=config["pretrained"], device=device
        )
        model.eval()
        return {"type": "mobileclip", "model": model, "preprocess": preprocess}

def encode_image(mb, img):
    if mb["type"] == "hf":
        inputs = mb["processor"](images=img, return_tensors="pt").to(device)
        with torch.no_grad():
            out = mb["model"].get_image_features(**inputs)
        feat = out if isinstance(out, torch.Tensor) else out.pooler_output
    else:
        tensor = mb["preprocess"](img).unsqueeze(0).to(device)
        with torch.no_grad():
            feat = mb["model"].encode_image(tensor)
    feat = feat / feat.norm(dim=-1, keepdim=True)
    return feat.cpu().numpy()[0]


# ============================================================
# CELL 2: Gold Standard laden
# ============================================================
GOLD_STANDARD_CSV = r"C:\Users\Lukas\masterthesis\thesis_2026\data\Annotation_Individual_CoEco_Landscape\01072026_annotations_Individual_CoEco_Landscape.csv"
IMAGE_DIR = Path(r"C:\Users\Lukas\masterthesis\thesis_2026\data\Annotation_Individual_CoEco_Landscape\30062026_random_sample_images_Individual_CoEco_Landscape")
CACHE_DIR = r"C:\Users\Lukas\masterthesis\thesis_2026\data\LinearProbe_AllModels_Comparison_IndividualCoEcoLandscape"
os.makedirs(CACHE_DIR, exist_ok=True)

df_gold = pd.read_csv(GOLD_STANDARD_CSV)

def norm(x):
    if x is None or pd.isna(x):
        return None
    x = str(x).strip().lower()
    if 'ecosystem' in x or 'community' in x:
        return 'community_ecosystem'
    if 'individual' in x:
        return 'individual'
    if 'landscape' in x:
        return 'landscape'
    return None

filenames, labels = [], []
for _, row in df_gold.iterrows():
    fname = row.iloc[0]
    gold  = norm(row['gold_standard'])
    if fname is None or gold is None:
        continue
    filenames.append(fname)
    labels.append(gold)

print(f"Geladen: {len(filenames)} Bilder mit Gold-Standard-Label")
print(pd.Series(labels).value_counts())

categories = ['individual', 'community_ecosystem', 'landscape']


# ============================================================
# CELL 3: Hilfsfunktion
# ============================================================
def safe_open(path):
    try:
        return Image.open(path).convert("RGB")
    except (UnidentifiedImageError, OSError) as e:
        print(f"  ⚠ Skipped ({os.path.basename(path)}): {e}")
        return None


# ============================================================
# CELL 4: Hauptschleife — pro Modell Linear Probe trainieren
# ============================================================
all_results = []

for config in MODEL_CONFIGS:
    model_name = config["name"]
    print(f"\n{'='*60}\nModell: {model_name}\n{'='*60}")

    model_bundle = load_model(config)

    # --- Embeddings extrahieren ---
    emb_path = os.path.join(CACHE_DIR, f"embeddings_{model_name}.npy")

    if os.path.exists(emb_path):
        print(f"  Lade gecachte Embeddings: {emb_path}")
        X = np.load(emb_path)
        y = np.array(labels)
    else:
        X, y, missing = [], [], []
        for fname, label in zip(filenames, labels):
            img_path = IMAGE_DIR / fname
            if not img_path.exists():
                missing.append(fname)
                continue
            img = safe_open(img_path)
            if img is None:
                continue
            emb = encode_image(model_bundle, img)
            X.append(emb)
            y.append(label)

        if missing:
            print(f"  WARNUNG: {len(missing)} Bilder nicht gefunden")

        X = np.array(X)
        y = np.array(y)
        np.save(emb_path, X)
        print(f"  Embeddings extrahiert und gecacht: {emb_path}")

    print(f"  Embeddings: {X.shape}")

    # --- Linear Probe + 5-fold CV ---
    clf = LogisticRegression(max_iter=1000, C=0.5, class_weight="balanced")
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

    auc_scores = cross_val_score(clf, X, y, cv=skf, scoring="roc_auc_ovr")
    y_pred     = cross_val_predict(clf, X, y, cv=skf)

    report = classification_report(y, y_pred, output_dict=True, zero_division=0)
    cm     = confusion_matrix(y, y_pred, labels=categories)

    print(f"  AUC OvR (5-fold CV): {auc_scores.mean():.3f} +/- {auc_scores.std():.3f}")
    print(classification_report(y, y_pred, target_names=sorted(categories), zero_division=0))
    print(f"  Confusion Matrix (individual / community_ecosystem / landscape):")
    print(cm)

    # --- Finales Modell speichern ---
    clf.fit(X, y)
    model_path = os.path.join(CACHE_DIR, f"linear_probe_{model_name}_individual_coeco_landscape.joblib")
    joblib.dump(clf, model_path)
    print(f"  Modell gespeichert: {model_path}")

    all_results.append({
        "Modell":           model_name,
        "Embedding_Dim":    X.shape[1],
        "AUC_OvR_mean":     round(auc_scores.mean(), 4),
        "AUC_OvR_std":      round(auc_scores.std(), 4),
        "Accuracy":         round(report["accuracy"], 4),
        "F1_individual":    round(report["individual"]["f1-score"], 4),
        "F1_coeco":         round(report["community_ecosystem"]["f1-score"], 4),
        "F1_landscape":     round(report["landscape"]["f1-score"], 4),
        "Macro_F1":         round(report["macro avg"]["f1-score"], 4),
    })

    del model_bundle
    torch.cuda.empty_cache()


# ============================================================
# CELL 5: Gesamtvergleich
# ============================================================
df_comparison = pd.DataFrame(all_results).sort_values("Macro_F1", ascending=False)
print("\n" + "="*60)
print("GESAMTVERGLEICH — Linear Probe pro Modell")
print("="*60)
print(df_comparison.to_string(index=False))

comparison_csv = os.path.join(CACHE_DIR, "linearprobe_model_comparison_individual_coeco_landscape.csv")
df_comparison.to_csv(comparison_csv, index=False)
print(f"\nGespeichert: {comparison_csv}")

Gerät: cuda
Geladen: 297 Bilder mit Gold-Standard-Label
landscape              211
community_ecosystem     60
individual              26
Name: count, dtype: int64

Modell: CLIP-ViT-B32


Loading weights: 100%|████████████████████████████████████████████████████████████| 398/398 [00:00<00:00, 18606.03it/s]


  Embeddings extrahiert und gecacht: C:\Users\Lukas\masterthesis\thesis_2026\data\LinearProbe_AllModels_Comparison_IndividualCoEcoLandscape\embeddings_CLIP-ViT-B32.npy
  Embeddings: (297, 512)
  AUC OvR (5-fold CV): 0.962 +/- 0.009
                     precision    recall  f1-score   support

community_ecosystem       0.67      0.73      0.70        60
         individual       0.77      0.88      0.82        26
          landscape       0.94      0.89      0.91       211

           accuracy                           0.86       297
          macro avg       0.79      0.84      0.81       297
       weighted avg       0.87      0.86      0.86       297

  Confusion Matrix (individual / community_ecosystem / landscape):
[[ 23   2   1]
 [  4  44  12]
 [  3  20 188]]
  Modell gespeichert: C:\Users\Lukas\masterthesis\thesis_2026\data\LinearProbe_AllModels_Comparison_IndividualCoEcoLandscape\linear_probe_CLIP-ViT-B32_individual_coeco_landscape.joblib

Modell: CLIP-ViT-L14


Loading weights: 100%|█████████████████████████████████████████████████████████████| 590/590 [00:00<00:00, 3058.97it/s]


  Embeddings extrahiert und gecacht: C:\Users\Lukas\masterthesis\thesis_2026\data\LinearProbe_AllModels_Comparison_IndividualCoEcoLandscape\embeddings_CLIP-ViT-L14.npy
  Embeddings: (297, 768)
  AUC OvR (5-fold CV): 0.951 +/- 0.012
                     precision    recall  f1-score   support

community_ecosystem       0.64      0.70      0.67        60
         individual       0.73      0.85      0.79        26
          landscape       0.94      0.89      0.91       211

           accuracy                           0.85       297
          macro avg       0.77      0.81      0.79       297
       weighted avg       0.86      0.85      0.85       297

  Confusion Matrix (individual / community_ecosystem / landscape):
[[ 22   3   1]
 [  6  42  12]
 [  2  21 188]]
  Modell gespeichert: C:\Users\Lukas\masterthesis\thesis_2026\data\LinearProbe_AllModels_Comparison_IndividualCoEcoLandscape\linear_probe_CLIP-ViT-L14_individual_coeco_landscape.joblib

Modell: EVA01-CLIP-g14
  Embeddings ext